In [1]:
from pathlib import Path

from rlmflow.utils.trace import load_trace

TRACE_PATH = Path("./generated/needle_trace/")

trace = load_trace(TRACE_PATH)
states = trace.states
events = trace.events
plot_mode = "events" if events else "snapshot"

print(f"loaded {len(states)} states from {TRACE_PATH}")
print(f"loaded {len(events)} node events")
print(f"metadata: {trace.metadata}")
print(f"final: {states[-1].agent_id} [{states[-1].type}] -> {getattr(states[-1], 'result', '')}")

loaded 9 states from generated/needle_trace
loaded 45 node events
metadata: {'answer': '84721'}
final: root [query] -> 


In [2]:
print(states[-1].transcript())

--- system ---
## Role

You are a recursive agent with a Python REPL. You solve tasks by writing and executing Python programs, and you can delegate subtasks to sub-agents with fresh context windows.

## REPL

- Every response is exactly one ```repl``` code block. Tools are already in the namespace.
- Variables persist across turns within one agent.
- `AGENT_ID`, `DEPTH`, `MAX_DEPTH` are set; cannot `delegate` when `DEPTH == MAX_DEPTH`.
- **Final answer:** call `done(answer)` exactly once when complete — that string is what the parent/user sees. No `done`, no result.
- **End the block after `wait`. Verify on the next turn.** The runtime won't stop you — if you call `done()` in the same block, it ends the agent right there with no verify turn. Instead, after `yield wait(...)` resumes, *return without calling `done()`*. The runtime then gives you a fresh turn (observation: `Children finished: ... / Generator resumed. Output: ...`) where you read files back / run / grep the artifact, and 

In [4]:
states[2].plot()

In [5]:
from rlmflow.utils.viewer import open_viewer

open_viewer(
    states,
    mode="snapshot",
    inline=True,
    height=720,
    quiet=True,
)

In [6]:
def show_step(i: int, *, transcript: bool = True):
    """Show one trace snapshot by index."""
    state = states[i]
    print(f"step {i} / {len(states) - 1}: {state.agent_id} [{state.type}]")
    print(state.plot("node_tree", states=states, events=events, step=i, mode=plot_mode))
    if transcript:
        print("\n--- transcript ---")
        print(state.transcript(include_system=False))
    return state.plot(states=states, events=events, step=i, mode=plot_mode, title=f"step {i} / {len(states) - 1}")

# Example:
show_step(0, transcript=False)

step 0 / 8: root [query]
root [query] {default}


In [12]:
def _scale_figure_elements(fig, mult: float) -> None:
    """Multiply pixel-sized elements (markers, lines, fonts) by ``mult``.

    The Plotly figure is laid out for ~420px-tall display. When we render to
    a much larger canvas for image export, those fixed pixel sizes become
    tiny relative to the canvas. Scale them up to match the canvas growth.
    """
    for trace in fig.data:
        marker = getattr(trace, "marker", None)
        if marker is not None:
            size = getattr(marker, "size", None)
            if isinstance(size, (list, tuple)):
                marker.size = [s * mult for s in size]
            elif isinstance(size, (int, float)):
                marker.size = size * mult
            line = getattr(marker, "line", None)
            if line is not None and getattr(line, "width", None) is not None:
                line.width = line.width * mult
        line = getattr(trace, "line", None)
        if line is not None and getattr(line, "width", None) is not None:
            line.width = line.width * mult
        textfont = getattr(trace, "textfont", None)
        if textfont is not None and getattr(textfont, "size", None) is not None:
            textfont.size = textfont.size * mult

    layout = fig.layout
    if layout.title and layout.title.font and layout.title.font.size:
        layout.title.font.size = layout.title.font.size * mult
    if layout.legend and layout.legend.font and layout.legend.font.size:
        layout.legend.font.size = layout.legend.font.size * mult
    if layout.font and layout.font.size:
        layout.font.size = layout.font.size * mult


def save_steps(
    states,
    path,
    *,
    fmt: str = "png",
    width: int = 1800,
    height: int = 1350,
    scale: float = 2.0,
    element_mult: float = 3.0,
    margin: dict | None = None,
):
    """Render every snapshot in ``states`` and save it as an image under ``path``.

    Each step is written as ``step_{i:02d}.{fmt}``. ``width``/``height`` set
    the logical canvas (defaults to 4:3). ``scale`` is a kaleido density
    multiplier for hi-dpi crispness. ``element_mult`` scales markers, edges,
    and fonts up so they don't look tiny on the larger export canvas — bump
    it for fatter nodes and bigger labels. ``margin`` tightens the plot's
    whitespace around the tree. Requires ``kaleido`` for image export —
    install with ``pip install -U kaleido``.
    """
    from pathlib import Path

    out_dir = Path(path)
    out_dir.mkdir(parents=True, exist_ok=True)
    margin = margin if margin is not None else {"l": 24, "r": 24, "t": 80, "b": 120}

    for i, state in enumerate(states):
        fig = state.plot(
            states=states,
            events=events,
            step=i,
            mode=plot_mode,
            title=f"step {i} / {len(states) - 1}: {state.agent_id} [{state.type}]",
        )
        _scale_figure_elements(fig, element_mult)
        fig.update_layout(margin=margin)
        out_path = out_dir / f"step_{i:02d}.{fmt}"
        fig.write_image(out_path, width=width, height=height, scale=scale)
        print(f"wrote {out_path}")

    return out_dir


save_steps(
    states,
    "./static/needle_trace_images",
    fmt="png",
    width=1800,
    height=1350,
    scale=2.0,
    element_mult=3.0,
    margin={"l": 24, "r": 24, "t": 80, "b": 120},
)

wrote static/needle_trace_images/step_00.png
wrote static/needle_trace_images/step_01.png
wrote static/needle_trace_images/step_02.png
wrote static/needle_trace_images/step_03.png
wrote static/needle_trace_images/step_04.png
wrote static/needle_trace_images/step_05.png
wrote static/needle_trace_images/step_06.png
wrote static/needle_trace_images/step_07.png
wrote static/needle_trace_images/step_08.png


PosixPath('static/needle_trace_images')

In [11]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    slider = widgets.IntSlider(
        value=0,
        min=0,
        max=len(states) - 1,
        step=1,
        description="step",
        continuous_update=False,
    )
    transcript_toggle = widgets.Checkbox(value=False, description="transcript")

    out = widgets.interactive_output(
        lambda i, transcript: display(show_step(i, transcript=transcript)),
        {"i": slider, "transcript": transcript_toggle},
    )
    display(widgets.VBox([widgets.HBox([slider, transcript_toggle]), out]))
except ImportError:
    print("ipywidgets is not installed. Use show_step(i) manually instead.")

In [ ]:
# Other useful views from the same node snapshots:

final = states[-1]

mermaid_flowchart = final.plot("flowchart")
sequence_diagram = final.plot("sequence")
gantt_html = final.plot("gantt", states=states)

print(mermaid_flowchart[:500])